# Washington, DC — Land Value Tax Shift Model

**Jurisdiction:** District of Columbia. DC is city, county and state in one, so there is a single real
property tax levy (no separate county, school or special-district levies). "Full stack" and "city levy"
are the same thing here.

**Reform modeled:** revenue-neutral **4:1 split-rate** (land taxed at 4× the building rate), solved
**separately within each DC tax class**, so the existing class differentials (Class 1 residential
$0.85, Class 2 commercial $1.65/$1.77/$1.89, Class 3 vacant $5.00, Class 4 blighted $10.00) and the
Class 2 value tiers are preserved. No burden moves between classes; only the land/building balance
inside each class changes.

**Relief kept:** homestead deduction ($91,950 in TY2026), senior/disabled 50% relief, the 10%
homestead assessment cap credit, and all full exemptions (federal, DC government, foreign government,
religious, educational, charitable, etc.).

**Data:** DC OTR *Integrated Tax System Public Extract* (ITSPE) from DC GIS, TY2026 billed values and
actual TY2026 tax bills.

| Concept | Column | Notes |
|---|---|---|
| Account / parcel id | `SSL` | Square-Suffix-Lot; condo units are their own accounts |
| Land value (TY2026 billed) | `PHASELAND` | `PHASELAND + PHASEBUILD == ASSESSMENT` for every record |
| Building value (TY2026 billed) | `PHASEBUILD` | |
| Total assessed value (TY2026) | `ASSESSMENT` | Pre-relief assessed value used for TY2026 bills |
| Proposed TY2027 values | `NEWLAND`, `NEWIMPR`, `NEWTOTAL` | Reported for reference only |
| Tax class | `CLASSTYPE` | 1A, 1B, 2, 3, 4 |
| Nominal rate ($/100) | `TAXRATE` | Class 2 rate is the bracket rate applied to the whole value |
| Mixed-use flag | `MIXEDUSE` | Split between Class 1 and Class 2 |
| Exemption / tax type | `MIX1TXTYPE` | Blank or `TX` = taxable; `US`, `DC`, `E0`–`E9`, `CE` = exempt |
| Homestead / senior code | `HSTDCODE` | 1 = homestead; 3, 5, 7 = senior/disabled-type relief |
| Current tax | `ANNUALTAX` | Actual TY2026 annual bill (after all relief) |
| Use code | `USECODE` | OTR land-use code; drives property categories |
| Ward | `PRMS_WARD` | Premise ward |
| Deleted record flag | `DELCODE` | `Y` = retired account, dropped |

**Assessment ratio:** 100% of market value. **Rate source:** OTR rate table (ITSPE PROPERTY TAX RATE
table, layer 64) and observed bills.

In [1]:
import sys
import os
from pathlib import Path

import geopandas as gpd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import load_dotenv

sys.path.insert(0, '../..')
sys.path.insert(0, '.')
REPO_ROOT = Path('../..').resolve()
load_dotenv(REPO_ROOT / '.env')

from lvt.lvt_utils import (
    model_split_rate_tax,
    calculate_category_tax_summary,
    print_category_tax_summary,
    save_standard_export,
)

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 30)

# Constants
CITY_NAME = 'washington_dc'
STATE_FIPS = '11'
COUNTY_FIPS = '001'
MODEL_TYPE = 'split_rate_4to1_within_class'
LAND_IMPROVEMENT_RATIO = 4.0

# Parcel-map export identity columns
PARCEL_ID_COL = 'SSL'
OWNER_NAME_COL = 'OWNERNAME'
OWNER_ADDRESS_COL = 'PREMISEADD'
# DC's Real Property Tax Database has no stable per-SSL deep link (search form is session-based).
PARCEL_URL_TEMPLATE = None

# Official revenue control: OCFO June 2026 revenue estimate, FY2026 real property tax ($ thousands 2,744,239)
OFFICIAL_REVENUE = 2_744_239_000
HOMESTEAD_DEDUCTION = 91_950  # TY2026, confirmed from bills below

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

## Section 2 — Load the tax roll and parcel geometry

In [2]:
# Step 2: Load cached downloads (fetch_dc.py pages DC GIS MapServer layers in parallel)
import fetch_dc

if not (DATA_DIR / 'itspe.parquet').exists():
    fetch_dc.fetch_layer(53, fetch_dc.ITSPE_FIELDS).to_parquet(DATA_DIR / 'itspe.parquet')
if not (DATA_DIR / 'condorelate.parquet').exists():
    fetch_dc.fetch_layer(52, ['SSL', 'MAT_SSL', 'REGIME', 'NAME', 'UNITNUM', 'OBJECTID']).to_parquet(
        DATA_DIR / 'condorelate.parquet')
if not (DATA_DIR / 'owner_polygons.gpq').exists():
    _p = fetch_dc.fetch_layer(40, ['SSL', 'UNDERLIES_CONDO', 'CONDOLOT', 'OBJECTID'], geometry=True)
    gpd.GeoDataFrame(_p, geometry='geometry', crs='EPSG:4326').to_parquet(DATA_DIR / 'owner_polygons.gpq')
if not (DATA_DIR / 'address_points.parquet').exists():
    fetch_dc.fetch_address_points()

raw = pd.read_parquet(DATA_DIR / 'itspe.parquet')
polys = gpd.read_parquet(DATA_DIR / 'owner_polygons.gpq')
condorelate = pd.read_parquet(DATA_DIR / 'condorelate.parquet')
addr_xref = pd.read_parquet(DATA_DIR / 'address_ssl_xref.parquet')
addr_pts = pd.read_parquet(DATA_DIR / 'address_points.parquet')

print(f"ITSPE records: {len(raw):,}  (deleted/retired: {(raw['DELCODE'] == 'Y').sum():,})")
print(f"Owner polygons: {len(polys):,}")
df = raw[raw['DELCODE'] == 'N'].copy().reset_index(drop=True)
print(f"Active tax accounts: {len(df):,}")
print("Bill year:", df['CY1YEAR'].value_counts().to_dict())

ITSPE records: 221,667  (deleted/retired: 5,697)
Owner polygons: 137,400


Active tax accounts: 215,970
Bill year: {'2026': 215970}


### Geometry: tax account → parcel polygon

Non-condo accounts match the Owner Polygons layer directly by SSL. Condo units are separate tax accounts
with no polygon of their own, so each is placed on its building's lot using, in order: (1) the
CONDORELATE master lot, (2) the unit's MAR address point (nearest owner polygon within 40 m), (3) any
sibling unit in the same condo regime or building address, (4) the condo master lot in the same square.
Geometry is used only for the census join and the map. Every tax figure is computed per account.

In [3]:
# Step 2b: Resolve each account to a map lot
poly_ssl = set(polys['SSL'])
cm = condorelate.drop_duplicates('SSL').set_index('SSL')
mat_from_condo = df['SSL'].map(cm['MAT_SSL'])
df['map_ssl'] = np.where(df['SSL'].isin(poly_ssl), df['SSL'],
                         np.where(mat_from_condo.isin(poly_ssl), mat_from_condo, None))
df['map_ssl'] = df['map_ssl'].astype(object).where(pd.notna(df['map_ssl']), None)
df['geo_source'] = np.where(df['SSL'].isin(poly_ssl), 'direct',
                            np.where(df['map_ssl'].notna(), 'condo_master', None))

# MAR address points -> nearest owner polygon
xref = addr_xref.dropna(subset=['MARID']).drop_duplicates('SSL').set_index('SSL')['MARID']
pts = addr_pts.dropna(subset=['LATITUDE']).drop_duplicates('MAR_ID').set_index('MAR_ID')
_m = df['map_ssl'].isna()
_marid = df.loc[_m, 'SSL'].map(xref)
_lat = _marid.map(pts['LATITUDE'])
_lon = _marid.map(pts['LONGITUDE'])
_ok = _lat.notna()
_pts = gpd.GeoDataFrame(index=_lat[_ok].index,
                        geometry=gpd.points_from_xy(_lon[_ok], _lat[_ok]), crs='EPSG:4326').to_crs(26985)
_j = gpd.sjoin_nearest(_pts, polys[['SSL', 'geometry']].to_crs(26985), max_distance=40, how='left')
_j = _j[~_j.index.duplicated()].dropna(subset=['SSL'])
df.loc[_j.index, 'map_ssl'] = _j['SSL']
df.loc[_j.index, 'geo_source'] = 'address_point'

# Sibling units in the same condo regime / building address
df['regime'] = df['SSL'].map(cm['REGIME'])
df['bldg_addr'] = (df['PREMISEADD'].str.replace(r'\s+#.*$', '', regex=True)
                   .str.replace(r'\s+WASHINGTON DC.*$', '', regex=True).str.strip())
for _key in ['regime', 'bldg_addr']:
    _known = (df[df['map_ssl'].notna() & df[_key].notna()]
              .groupby(_key)['map_ssl'].agg(lambda s: s.mode().iat[0]))
    _fill = df['map_ssl'].isna() & df[_key].isin(_known.index)
    df.loc[_fill, 'map_ssl'] = df.loc[_fill, _key].map(_known)
    df.loc[_fill, 'geo_source'] = f'sibling_{_key}'

# Condo master lot (or largest lot) in the same square
_sq_poly = polys.assign(sq=polys['SSL'].str[:8].str.strip(),
                        area=polys.to_crs(26985).area,
                        condo=polys['UNDERLIES_CONDO'].fillna(0))
_sq_best = (_sq_poly.sort_values(['condo', 'area'], ascending=False)
            .drop_duplicates('sq').set_index('sq')['SSL'])
_fill = df['map_ssl'].isna()
df.loc[_fill, 'map_ssl'] = df.loc[_fill, 'SSL'].str[:8].str.strip().map(_sq_best)
df.loc[_fill & df['map_ssl'].notna(), 'geo_source'] = 'square_fallback'
print(df['geo_source'].fillna('unplaced').value_counts())

geo_source
direct               131695
condo_master          54296
address_point         21609
square_fallback        3669
sibling_bldg_addr      2531
sibling_regime         1988
unplaced                182
Name: count, dtype: int64


## Section 3 — Classify accounts

**Fully exempt:** any account whose tax type is an exemption code (`US` federal, `DC` District,
`E0`–`E9`, `CE`), or whose TY2026 bill is $0 (abatements, special acts, zero-value records). These
are held out of the solver and the charts and are reported by count and value.

**Tax-class pools for the solver:** `1A`, `1B`, `2`, `3`, `4`, plus `MU` for the split
residential/commercial mixed-use accounts (their bill blends two classes).

In [4]:
# Step 3a: Exemption flag and tax-class pool
EXEMPT_TYPES = {'US', 'DC', 'CE', 'E0', 'E1', 'E2', 'E3', 'E4', 'E5', 'E6', 'E7', 'E8', 'E9', 'RL', 'HP'}
df['tax_type'] = np.where(df['MIXEDUSE'] == 'Y', 'MU', df['MIX1TXTYPE'].fillna('TX'))
df['full_exmp'] = (df['tax_type'].isin(EXEMPT_TYPES) | (df['ANNUALTAX'].fillna(0) <= 0)
                   | (df['ASSESSMENT'].fillna(0) <= 0) | (df['CY1YEAR'] != '2026')).astype(int)
df['tax_pool'] = np.where(df['MIXEDUSE'] == 'Y', 'MU', df['CLASSTYPE'].fillna('NA'))

EXEMPT_LABELS = {
    'US': 'Federal government', 'DC': 'DC government', 'E6': 'Foreign government (embassies)',
    'E2': 'Educational', 'E1': 'Religious', 'E8': 'Special acts of Congress / misc.',
    'E3': 'Charitable', 'E4': 'Hospitals', 'E0': 'Low-income housing', 'E9': 'WMATA',
    'E7': 'Cemeteries', 'E5': 'Libraries', 'CE': 'Other exempt (CE)',
}
df['exempt_group'] = np.where(df['full_exmp'] == 1,
                              df['tax_type'].map(EXEMPT_LABELS).fillna('Taxable record with $0 bill'),
                              None)
print(df['full_exmp'].value_counts().rename({0: 'taxable', 1: 'fully exempt'}))
print(df.loc[df['full_exmp'] == 0, 'tax_pool'].value_counts())

full_exmp
taxable         204347
fully exempt     11623
Name: count, dtype: int64
tax_pool
1B    160656
1A     33083
2       7977
3       1663
MU       758
4        197
NA        13
Name: count, dtype: int64


In [5]:
# Step 3b: Property categories from OTR use codes
def categorize_dc(usecode: str, mixed: str) -> str:
    if mixed == 'Y':
        return 'Mixed Use'
    if usecode is None or pd.isna(usecode):
        return 'Other'
    u = str(usecode).zfill(3)
    if u == '011':
        return 'Townhome / Rowhouse'
    if u in {'001', '012', '013', '019'}:
        return 'Single Family Residential'
    if u in {'016', '017', '116', '117', '316', '516', '216', '217'}:
        return 'Condominium'
    if u in {'023', '024'}:
        return 'Small Multi-Family (2-4 units)'
    if u in {'021', '022', '025', '029', '002', '026', '027', '028'}:
        return 'Large Multi-Family (5+ units)'
    if u in {'014', '018', '214', '416', '417'}:
        return 'Other Residential'
    if u in {'015', '126', '127'}:
        return 'Mixed Use'
    if u in {'031', '032', '033', '035', '037', '039', '003'}:
        return 'Hotel'
    if u in {'041', '042', '043', '044', '045', '046', '047', '049', '067', '068', '004'}:
        return 'Retail / General Commercial'
    if u in {'048', '051', '052', '053', '056', '057', '058', '059', '061', '005'}:
        return 'Office / Commercial Condo'
    if u in {'063', '064', '096', '196'}:
        return 'Transportation - Parking'
    if u.startswith('07') or u == '007':
        return 'Industrial'
    if u in {'091', '092', '093', '094', '095', '097', '191', '192', '193', '194', '195', '197'}:
        return 'Vacant Land'
    if u in {'034', '036', '038', '062', '065', '066', '069', '165', '265', '365', '465', '006'}:
        return 'Other Commercial'
    if u.startswith('08') or u in {'189', '008'}:
        return 'Other'  # taxable institutional / special purpose
    return 'Other'


df['PROPERTY_CATEGORY'] = [categorize_dc(u, m) for u, m in zip(df['USECODE'], df['MIXEDUSE'])]
# Override: $0 building value on a land-bearing account -> Vacant Land
df.loc[(df['PHASEBUILD'] <= 0) & (df['PHASELAND'] > 0), 'PROPERTY_CATEGORY'] = 'Vacant Land'
tx = df[df['full_exmp'] == 0]
print(tx['PROPERTY_CATEGORY'].value_counts())
print("\nUse codes inside 'Other' / 'Other Residential':")
print(tx[tx['PROPERTY_CATEGORY'].isin(['Other', 'Other Residential'])]
      .groupby(['PROPERTY_CATEGORY', 'PROPTYPE']).size().sort_values(ascending=False).head(15))

PROPERTY_CATEGORY
Condominium                       66407
Single Family Residential         47790
Townhome / Rowhouse               44953
Other Residential                 14604
Small Multi-Family (2-4 units)     8927
Vacant Land                        7080
Large Multi-Family (5+ units)      6913
Retail / General Commercial        3266
Office / Commercial Condo          1929
Mixed Use                           777
Other                               457
Transportation - Parking            380
Industrial                          355
Other Commercial                    270
Hotel                               239
Name: count, dtype: int64

Use codes inside 'Other' / 'Other Residential':
PROPERTY_CATEGORY  PROPTYPE                      
Other Residential  Residential-Condominium (Garag    13945
                   Residential-Garage                  566
Other              Religious                           135
                   Public Service                       99
                   Sp

### Tax base summary (the crib-sheet numbers)

Values are TY2026 billed assessments (`PHASELAND`, `PHASEBUILD`). "Taxable value after relief" is the
value that, at each account's nominal class rate, reproduces its actual bill (assessed value less the
homestead deduction, the cap credit and senior/disabled relief, expressed in value terms).

In [6]:
# Step 3c: Tax base summary tables
def nominal_rate(row_class: pd.Series, assess: pd.Series, taxrate: pd.Series) -> pd.Series:
    # Class 1A/1B: $0.85 on the first $2.5M, $1.00 above; others: bracket rate as billed
    res = row_class.isin(['1A', '1B'])
    tiered = np.where(assess > 2_500_000, (0.85 * 2_500_000 + 1.00 * (assess - 2_500_000)) / assess, 0.85)
    return pd.Series(np.where(res, tiered, taxrate), index=assess.index)


df['nominal_rate'] = nominal_rate(df['CLASSTYPE'], df['ASSESSMENT'], df['TAXRATE'])
df['eff_rate'] = np.where(df['ASSESSMENT'] > 0, df['ANNUALTAX'] / df['ASSESSMENT'] * 100, 0.0)

CLASS_LABELS = {
    '1A': 'Class 1A: Residential, 3+ units ($0.85)',
    '1B': 'Class 1B: Residential, 1-2 units ($0.85)',
    '2': 'Class 2: Commercial ($1.65/$1.77/$1.89)',
    '3': 'Class 3: Vacant ($5.00)',
    '4': 'Class 4: Blighted ($10.00)',
    'MU': 'Mixed use (Class 1 + Class 2)',
}


def base_table(d: pd.DataFrame, by: str) -> pd.DataFrame:
    g = d.groupby(by).agg(
        accounts=('SSL', 'size'),
        land_value=('PHASELAND', 'sum'),
        building_value=('PHASEBUILD', 'sum'),
        assessed_value=('ASSESSMENT', 'sum'),
        tax_ty2026=('ANNUALTAX', 'sum'),
    )
    g['land_share'] = g['land_value'] / g['assessed_value']
    g['effective_rate_per_100'] = g['tax_ty2026'] / g['assessed_value'] * 100
    g['share_of_levy'] = g['tax_ty2026'] / g['tax_ty2026'].sum()
    return g.sort_values('assessed_value', ascending=False)


taxable = df[df['full_exmp'] == 0].copy()
exempt = df[df['full_exmp'] == 1].copy()

base_by_class = base_table(taxable.assign(cls=taxable['tax_pool'].map(CLASS_LABELS)), 'cls')
base_by_cat = base_table(taxable, 'PROPERTY_CATEGORY')
exempt_by_type = base_table(exempt, 'exempt_group').drop(columns=['tax_ty2026', 'effective_rate_per_100', 'share_of_levy'])

_all = df.assign(status=np.where(df['full_exmp'] == 1, 'Fully exempt', 'Taxable'))
overall = _all.groupby('status')[['PHASELAND', 'PHASEBUILD', 'ASSESSMENT', 'NEWTOTAL', 'ANNUALTAX']].sum()
overall.loc['All property'] = overall.sum()
overall['land_share'] = overall['PHASELAND'] / overall['ASSESSMENT']

_fmt = lambda t: t.style.format({c: '${:,.0f}' for c in t.columns if 'value' in c or 'tax' in c})
print("Overall (TY2026 billed; NEWTOTAL = proposed TY2027):")
print((overall / 1e9).round(2).assign(land_share=overall['land_share'].round(3)))
print("\nTaxable base by DC tax class ($B):")
print(base_by_class.assign(**{c: base_by_class[c] / 1e9 for c in ['land_value', 'building_value', 'assessed_value', 'tax_ty2026']}).round(3))
print("\nTaxable base by property category ($B):")
print(base_by_cat.assign(**{c: base_by_cat[c] / 1e9 for c in ['land_value', 'building_value', 'assessed_value', 'tax_ty2026']}).round(3))
print("\nExempt property by type ($B):")
print(exempt_by_type.assign(**{c: exempt_by_type[c] / 1e9 for c in ['land_value', 'building_value', 'assessed_value']}).round(3))

Overall (TY2026 billed; NEWTOTAL = proposed TY2027):
              PHASELAND  PHASEBUILD  ASSESSMENT  NEWTOTAL  ANNUALTAX  land_share
status                                                                          
Fully exempt      68.37       42.66      111.03    117.85       0.00       0.616
Taxable          107.03      153.87      260.90    266.05       2.84       0.410
All property     175.41      196.52      371.93    383.91       2.85       0.472

Taxable base by DC tax class ($B):
                                          accounts  land_value  building_value  assessed_value  tax_ty2026  land_share  effective_rate_per_100  share_of_levy
cls                                                                                                                                                          
Class 1B: Residential, 1-2 units ($0.85)    160656      57.230          73.308         130.538       0.977       0.438                   0.748          0.344
Class 2: Commercial ($1.65/$1.77

In [7]:
# Step 3d: Residential relief — homestead, senior/disabled, assessment cap
res = taxable[taxable['tax_pool'].isin(['1A', '1B'])]
relief = res.assign(
    relief_type=res['HSTDCODE'].map({'1': 'Homestead deduction', '5': 'Senior/disabled 50% relief',
                                     '3': 'Senior/disabled 50% relief (other)', '7': 'Senior/disabled 50% relief (other)'})
    .fillna('No homestead/senior relief'),
    nominal_tax=res['ASSESSMENT'] * res['nominal_rate'] / 100,
)
relief_tbl = relief.groupby('relief_type').agg(
    accounts=('SSL', 'size'), assessed_value=('ASSESSMENT', 'sum'),
    nominal_tax=('nominal_tax', 'sum'), actual_tax=('ANNUALTAX', 'sum'))
relief_tbl['relief_dollars'] = relief_tbl['nominal_tax'] - relief_tbl['actual_tax']
relief_tbl['median_bill'] = relief.groupby('relief_type')['ANNUALTAX'].median()
print(relief_tbl.round(0))
_h = res[res['HSTDCODE'] == '1']
_implied = (_h['ASSESSMENT'] - _h['ANNUALTAX'] / (_h['nominal_rate'] / 100)).round(-1)
print("Note: relief in the no-homestead row is mostly a uniform ~$16,000-of-value reduction on ~35k condo\n"
      "bills (source not identified in OTR docs; it is in the actual bills, so the model carries it).")
print(f"\nMost common implied homestead deduction: ${_implied.mode().iat[0]:,.0f} "
      f"(constant HOMESTEAD_DEDUCTION = ${HOMESTEAD_DEDUCTION:,})")

                                    accounts  assessed_value  nominal_tax   actual_tax  relief_dollars  median_bill
relief_type                                                                                                        
Homestead deduction                    80883    7.284876e+10  622178986.0  550259146.0      71919840.0       5402.0
No homestead/senior relief             95027    8.418195e+10  757958397.0  696379187.0      61579210.0       3720.0
Senior/disabled 50% relief             16952    1.132733e+10   96341424.0   35247592.0      61093832.0       1639.0
Senior/disabled 50% relief (other)       877    5.615738e+08    4774989.0    1768320.0       3006669.0       1646.0


Note: relief in the no-homestead row is mostly a uniform ~$16,000-of-value reduction on ~35k condo
bills (source not identified in OTR docs; it is in the actual bills, so the model carries it).

Most common implied homestead deduction: $91,950 (constant HOMESTEAD_DEDUCTION = $91,950)


## Section 4 — Current tax and revenue check

**Current tax = the actual TY2026 bill** (`ANNUALTAX`), the Baltimore pattern. Rebuilding bills from
rates would have to reproduce the cap credit, senior relief and partial abatements parcel by parcel.
The observed bill already contains all of them.

**Taxable land and building after relief:** each account's relief is spread proportionally across land
and building (`taxable = billed value × actual bill ÷ nominal-rate bill`). The library default puts
dollar exemptions on the building first. Here that would turn senior/disabled tax relief (a credit
against tax, not a value deduction) into land-only bases and show large artificial increases for
seniors. With proportional allocation each household keeps the same *share* of relief under the reform.

In [8]:
# Step 4: Current tax
taxable['current_tax'] = taxable['ANNUALTAX'].astype(float)
_relief_factor = (taxable['eff_rate'] / taxable['nominal_rate']).clip(upper=1.0)
_relief_factor = np.where(taxable['tax_pool'] == 'MU', 1.0, _relief_factor)
taxable['taxable_land_value'] = taxable['PHASELAND'] * _relief_factor
taxable['taxable_improvement_value'] = taxable['PHASEBUILD'] * _relief_factor

current_revenue = taxable['current_tax'].sum()
gap_pct = (current_revenue / OFFICIAL_REVENUE - 1) * 100
print(f"Modeled (sum of TY2026 bills): ${current_revenue:,.0f}")
print(f"Official (OCFO FY2026 real property tax estimate, June 2026): ${OFFICIAL_REVENUE:,.0f}")
print(f"Gap: {gap_pct:+.2f}%")
assert abs(gap_pct) < 5.0, f"Revenue gap {gap_pct:.1f}% exceeds threshold"

Modeled (sum of TY2026 bills): $2,844,668,282
Official (OCFO FY2026 real property tax estimate, June 2026): $2,744,239,000
Gap: +3.66%


The levy (sum of bills) runs about 4% above the OCFO's FY2026 *cash revenue* estimate. That is expected:
the OCFO figure is net of commercial assessment-appeal refunds (which the OCFO says are reducing FY2026
revenue), uncollected amounts, and real property tax dedicated outside the general fund (e.g. TIF/PILOT
debt service). The modeled base is the full billed levy.

## Section 5 — 4:1 split-rate, revenue-neutral within each tax class

Within each pool, the solver works on **rate-weighted** land and building values
(`value × effective rate ÷ pool base rate`). This makes each account's current bill exactly
`pool rate × (weighted land + weighted building)`, so Class 2's $1.77/$1.89 tiers, the Class 1
$1.00 tier above $2.5M, and every account's relief carry through. The solver then finds the pool's land
and building rates (land = 4 × building) that raise the same revenue as today.

In [9]:
# Step 5: Solve per tax-class pool
POOL_BASE_RATE = {'1A': 0.85, '1B': 0.85, '2': 1.65, '3': 5.00, '4': 10.00, 'MU': 1.65}
taxable['w_land'] = taxable['PHASELAND'] * taxable['eff_rate'] / taxable['tax_pool'].map(POOL_BASE_RATE)
taxable['w_bldg'] = taxable['PHASEBUILD'] * taxable['eff_rate'] / taxable['tax_pool'].map(POOL_BASE_RATE)

pool_results, pool_rates = [], []
for pool, sub in taxable.groupby('tax_pool'):
    target = sub['current_tax'].sum()
    if pool not in POOL_BASE_RATE:
        # 13 billed accounts with no tax class on the extract: held at current tax
        out = sub.copy()
        out['new_tax'] = out['current_tax']
        pool_results.append(out)
        print(f"Pool {pool}: {len(sub)} accounts held unchanged (${target:,.0f})")
        continue
    lm, im, rev, out = model_split_rate_tax(
        df=sub.copy(), land_value_col='w_land', improvement_value_col='w_bldg',
        current_revenue=target, land_improvement_ratio=LAND_IMPROVEMENT_RATIO)
    pool_results.append(out)
    pool_rates.append({'pool': pool, 'label': CLASS_LABELS.get(pool, pool), 'accounts': len(sub),
                       'revenue': target, 'base_rate_per_100': POOL_BASE_RATE[pool],
                       'land_rate_per_100': lm / 10, 'building_rate_per_100': im / 10,
                       'new_revenue': rev})
gdf_model = pd.concat(pool_results).sort_index()
rates_tbl = pd.DataFrame(pool_rates).set_index('pool')
print(rates_tbl.round(4))

gdf_model['tax_change'] = gdf_model['new_tax'] - gdf_model['current_tax']
gdf_model['tax_change_pct'] = gdf_model['tax_change'] / gdf_model['current_tax'] * 100

# Effective citywide land / building millage (per $1,000 of taxable value after relief) for the export
land_tax = (gdf_model['w_land'] * gdf_model['tax_pool'].map(rates_tbl['land_rate_per_100']) / 100).sum()
bldg_tax = (gdf_model['w_bldg'] * gdf_model['tax_pool'].map(rates_tbl['building_rate_per_100']) / 100).sum()
land_millage = land_tax / gdf_model['taxable_land_value'].sum() * 1000
improvement_millage = bldg_tax / gdf_model['taxable_improvement_value'].sum() * 1000
n_exempt = int(df['full_exmp'].sum())
print(f"\nHeld out {n_exempt:,} fully-exempt accounts (excluded from model, export, and charts).")
print(f"Revenue: current ${gdf_model['current_tax'].sum():,.0f}  new ${gdf_model['new_tax'].sum():,.0f}")
print(f"Effective citywide millage — land {land_millage:.3f}, building {improvement_millage:.3f}")

category_summary = calculate_category_tax_summary(
    df=gdf_model, category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax', new_tax_col='new_tax')
print_category_tax_summary(category_summary, title='Washington, DC — 4:1 split-rate within class')

Pool NA: 13 accounts held unchanged ($2,628,516)


                                         label  accounts       revenue  base_rate_per_100  land_rate_per_100  building_rate_per_100   new_revenue
pool                                                                                                                                             
1A     Class 1A: Residential, 3+ units ($0.85)     33083  3.070640e+08               0.85             1.8412                 0.4603  3.070640e+08
1B    Class 1B: Residential, 1-2 units ($0.85)    160656  9.765903e+08               0.85             1.4769                 0.3692  9.765903e+08
2      Class 2: Commercial ($1.65/$1.77/$1.89)      7977  1.365305e+09               1.65             2.8025                 0.7006  1.365305e+09
3                      Class 3: Vacant ($5.00)      1663  6.567865e+07               5.00             7.6125                 1.9031  6.567865e+07
4                   Class 4: Blighted ($10.00)       197  1.151600e+07              10.00            14.1065                


Washington, DC — 4:1 split-rate within class
                      Category  Count Total Tax Δ ($) Total Δ (%) Mean Δ ($) Median Δ ($) Avg % Δ Median % Δ % Parcels > +10% % Parcels < -10%
                   Condominium  66407    $-42,477,286      -16.2%      $-640        $-543  -15.8%     -17.5%             2.5%            93.9%
     Single Family Residential  47790      $7,510,111        1.9%       $157         $298    6.6%       6.6%            41.8%            15.4%
           Townhome / Rowhouse  44953     $30,253,581        9.8%       $673         $778   13.1%      15.0%            59.8%            10.3%
             Other Residential  14604        $782,405       14.3%        $54           $6    5.6%       2.9%             7.2%             1.8%
Small Multi-Family (2-4 units)   8927      $3,158,318        3.6%       $354         $577    6.8%       7.0%            40.3%            10.4%
                   Vacant Land   7080     $43,049,708       67.2%     $6,080         $475  106.2

In [10]:
# Step 5b: Artifact scan (validate.md Gate 5)
d = gdf_model.copy()
d['bldg_share'] = d['PHASEBUILD'] / d['ASSESSMENT'].replace(0, np.nan)
scan = d.groupby('PROPERTY_CATEGORY').agg(
    n=('tax_change_pct', 'size'),
    median_pct=('tax_change_pct', 'median'),
    median_bldg_share=('bldg_share', 'median'),
    pct_zero_bldg=('PHASEBUILD', lambda s: (s <= 1000).mean()),
    current_tax=('current_tax', 'sum'),
    new_tax=('new_tax', 'sum'),
).sort_values('median_pct', ascending=False)
scan['total_change_pct'] = (scan['new_tax'] / scan['current_tax'] - 1) * 100
print(scan.round(3))

                                    n  median_pct  median_bldg_share  pct_zero_bldg   current_tax       new_tax  total_change_pct
PROPERTY_CATEGORY                                                                                                                
Vacant Land                      7080     116.612              0.000          0.983  6.405453e+07  1.071042e+08            67.208
Transportation - Parking          380      67.623              0.018          0.137  1.531075e+07  2.198972e+07            43.623
Other Commercial                  270      38.802              0.244          0.004  1.685975e+07  1.870401e+07            10.939
Industrial                        355      22.533              0.368          0.008  2.865113e+07  3.256969e+07            13.677
Large Multi-Family (5+ units)    6913      16.966              0.607          0.000  2.742382e+08  2.578105e+08            -5.990
Mixed Use                         777      15.139              0.650          0.004  1.153

### Gate 5 finding: condo land share is an OTR convention

The scan shows condos (and the Class 1A/1B medians in condo-heavy Wards 1 and 2) pinned at one exact
value (about -17.5%). That is the outcome for a 30% land share. OTR books ~98% of owner-occupied condo
units (use codes 016/017/018) at **exactly 30% land / 70% building**, so the per-unit land signal is
an assessor convention, not a market estimate. The cell below checks it against an independent
estimate: unit land area × the neighborhood's median land $/sq ft for non-condo comparables (row and
flat conversions for horizontal condos; apartment buildings for vertical condos).

**Decision:** keep OTR's official values. Comps put low-rise condos near 30% (median ≈31%). For
high-rise condos the comps give a lower land share (median ≈14%), so if anything the modeled condo
savings are **conservative**. The imputation is too noisy to replace the official split. Treat condo
results as a category-level average, not per-unit precision.

In [11]:
# Step 5c: Condo land-share check
_lpsf = (taxable['PHASELAND'] / taxable['LANDAREA'].replace(0, np.nan))
_t = taxable.assign(lpsf=_lpsf, land_share=taxable['PHASELAND'] / taxable['ASSESSMENT'])
for _code, _label in [('016', 'Horizontal condo'), ('017', 'Vertical condo'), ('018', 'Garage condo')]:
    _s = _t.loc[_t['USECODE'] == _code, 'land_share']
    print(f"{_label:17s} n={len(_s):6,}  exactly 30% land: {(_s.sub(0.30).abs() < 0.002).mean():.1%}")
for _codes, _comps, _label in [(['016'], ['011', '023', '024', '013'], 'Horizontal condo vs row/flat comps'),
                               (['017'], ['021', '022', '025', '029', '216', '217'], 'Vertical condo vs apartment comps')]:
    _o = _t[_t['USECODE'].isin(_codes)]
    _c = _t[_t['USECODE'].isin(_comps)]
    _rate = _o['SUBNBHD'].map(_c.groupby('SUBNBHD')['lpsf'].median()).fillna(
        _o['NBHD'].map(_c.groupby('NBHD')['lpsf'].median()))
    _imp = (_o['LANDAREA'] * _rate).clip(upper=_o['ASSESSMENT'] * 0.9) / _o['ASSESSMENT']
    print(f"{_label}: imputed land share p25/median/p75 = "
          f"{_imp.quantile(.25):.0%} / {_imp.median():.0%} / {_imp.quantile(.75):.0%}  (OTR: 30%)")

Horizontal condo  n=30,105  exactly 30% land: 98.2%
Vertical condo    n=30,319  exactly 30% land: 97.5%
Garage condo      n=13,988  exactly 30% land: 92.8%
Horizontal condo vs row/flat comps: imputed land share p25/median/p75 = 19% / 31% / 61%  (OTR: 30%)


Vertical condo vs apartment comps: imputed land share p25/median/p75 = 6% / 14% / 32%  (OTR: 30%)


## Section 6 — Crib-sheet tables (ward, homeowner, bill size)

In [12]:
# Step 6a: Outcome tables
def outcome(d: pd.DataFrame, by: str) -> pd.DataFrame:
    g = d.groupby(by).agg(
        accounts=('SSL', 'size'),
        current_tax=('current_tax', 'sum'),
        new_tax=('new_tax', 'sum'),
        median_change_pct=('tax_change_pct', 'median'),
        median_change_dollars=('tax_change', 'median'),
        share_pay_less=('tax_change', lambda s: (s < 0).mean()),
    )
    g['total_change_pct'] = (g['new_tax'] / g['current_tax'] - 1) * 100
    return g


gdf_model['ward'] = gdf_model['PRMS_WARD'].fillna('Unknown').astype(str).str.replace('WARD ', 'Ward ', case=False)
gdf_model['owner_occ'] = np.where(gdf_model['HSTDCODE'].isin(['1', '3', '5', '7']),
                                  'Owner-occupied (homestead/senior)', 'Not owner-occupied')
by_class = outcome(gdf_model.assign(cls=gdf_model['tax_pool'].map(CLASS_LABELS)), 'cls')
by_cat = outcome(gdf_model, 'PROPERTY_CATEGORY').sort_values('median_change_pct')
by_ward = outcome(gdf_model, 'ward')
by_ward_res = outcome(gdf_model[gdf_model['tax_pool'] == '1B'], 'ward')
by_owner = outcome(gdf_model[gdf_model['tax_pool'] == '1B'], 'owner_occ')
homes = gdf_model[gdf_model['PROPERTY_CATEGORY'].isin(
    ['Single Family Residential', 'Townhome / Rowhouse', 'Condominium', 'Small Multi-Family (2-4 units)'])]
homes = homes.assign(value_quintile=pd.qcut(homes['ASSESSMENT'], 5,
                                            labels=['Q1 (lowest value)', 'Q2', 'Q3', 'Q4', 'Q5 (highest value)']))
by_home_quintile = outcome(homes, 'value_quintile')
for name, t in [('class', by_class), ('category', by_cat), ('ward (all taxable)', by_ward),
                ('ward (Class 1B homes)', by_ward_res), ('owner occupancy (Class 1B)', by_owner),
                ('home value quintile', by_home_quintile)]:
    print(f"\nBy {name}:")
    print(t.round(3))


By class:
                                          accounts   current_tax       new_tax  median_change_pct  median_change_dollars  share_pay_less  total_change_pct
cls                                                                                                                                                       
Class 1A: Residential, 3+ units ($0.85)      33083  3.070640e+08  3.070640e+08              2.891                  8.599           0.177               0.0
Class 1B: Residential, 1-2 units ($0.85)    160656  9.765903e+08  9.765903e+08             -4.414               -136.407           0.552               0.0
Class 2: Commercial ($1.65/$1.77/$1.89)       7977  1.365305e+09  1.365305e+09              4.862                183.288           0.461              -0.0
Class 3: Vacant ($5.00)                       1663  6.567865e+07  6.567865e+07              1.858                421.554           0.465               0.0
Class 4: Blighted ($10.00)                     197  1.15160

In [13]:
# Step 6b: Save crib-sheet tables
TABLE_DIR = REPO_ROOT / 'analysis' / 'reports' / CITY_NAME / 'tables'
TABLE_DIR.mkdir(parents=True, exist_ok=True)
for name, t in [('overall', overall), ('base_by_class', base_by_class), ('base_by_category', base_by_cat),
                ('exempt_by_type', exempt_by_type), ('relief', relief_tbl), ('split_rates', rates_tbl),
                ('outcome_by_class', by_class), ('outcome_by_category', by_cat), ('outcome_by_ward', by_ward),
                ('outcome_by_ward_1b', by_ward_res), ('outcome_by_owner_1b', by_owner),
                ('outcome_by_home_quintile', by_home_quintile), ('artifact_scan', scan)]:
    t.to_csv(TABLE_DIR / f'{name}.csv')
_keep = ['SSL', 'PREMISEADD', 'OWNERNAME', 'USECODE', 'PROPTYPE', 'tax_pool', 'PROPERTY_CATEGORY', 'ward', 'owner_occ',
         'HSTDCODE', 'PHASELAND', 'PHASEBUILD', 'ASSESSMENT', 'current_tax', 'new_tax', 'tax_change', 'tax_change_pct']
gdf_model[_keep].to_parquet(DATA_DIR / 'model_results.parquet')
print(f"Saved tables to {TABLE_DIR}")

Saved tables to /home/user/LVTShift/analysis/reports/washington_dc/tables


## Section 7 — Census join, standard export, report, parcel map

Census join uses each account's map-lot centroid (condo units take their building's lot). Accounts with
no placed lot get no block group. The parcel map shows one feature per lot: condo units on the same lot
are summed into one building-level feature so units don't stack.

In [14]:
# Attach map-lot geometry to every modeled account
_lot_geom = polys.drop_duplicates('SSL').set_index('SSL')['geometry']
gdf_model['geometry'] = gdf_model['map_ssl'].map(_lot_geom)
df = gpd.GeoDataFrame(gdf_model, geometry='geometry', crs='EPSG:4326')
_nogeom = df['geometry'].isna()
print(f"Accounts without geometry: {_nogeom.sum():,} ({df.loc[_nogeom, 'current_tax'].sum() / 1e6:,.1f}M of tax)")
df_geo = df[~_nogeom].copy()
df_nogeo = df[_nogeom].copy()
df = df_geo

Accounts without geometry: 0 (0.0M of tax)


In [15]:
# Census join — OPTIONAL, must happen before export
import os
import concurrent.futures
from lvt.census_utils import get_census_data_with_boundaries, match_to_census_blockgroups

_fips = STATE_FIPS + COUNTY_FIPS
_census_cols = ['std_geoid', 'median_income', 'minority_pct', 'black_pct']
if not os.getenv('CENSUS_API_KEY'):
    # No key configured — the census join is optional, so skip it cleanly.
    # The model still produces the three core tax-impact charts; only the four
    # neighborhood-equity charts are skipped. This is expected, not a failure.
    print("Census join: skipped (no CENSUS_API_KEY)")
    for _col in _census_cols:
        df[_col] = float('nan')
else:
    try:
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as _ex:
            _future = _ex.submit(get_census_data_with_boundaries, _fips, 2022)
            try:
                census_data, census_gdf = _future.result(timeout=90)
                df = match_to_census_blockgroups(df, census_gdf)
                # Merge demographic attributes onto df by std_geoid
                _demo_cols = ['std_geoid', 'median_income', 'minority_pct', 'black_pct', 'total_pop']
                df = df.merge(
                    census_data[[c for c in _demo_cols if c in census_data.columns
                                  and (c == 'std_geoid' or c not in df.columns)]],
                    on='std_geoid', how='left'
                )
                print(f"Census join: {df['std_geoid'].notna().mean()*100:.1f}% matched")
            except concurrent.futures.TimeoutError:
                print("Census API timed out — skipping census join")
                for _col in _census_cols:
                    df[_col] = float('nan')
    except Exception as e:
        print(f"Census join failed: {e}")
        for _col in _census_cols:
            df[_col] = float('nan')

# Re-attach the few accounts with no geometry so the export covers the full levy
df = pd.concat([df, df_nogeo], ignore_index=True)

Census join: skipped (no CENSUS_API_KEY)


In [16]:
# Export — df must have census columns by this point
from lvt.lvt_utils import save_standard_export
out_df = save_standard_export(
    df=df,
    city=CITY_NAME,
    output_path=f'../../analysis/data/{CITY_NAME}.csv',
    model_type=MODEL_TYPE,
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    property_category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
    tax_change_col='tax_change',
    tax_change_pct_col='tax_change_pct',
    taxable_land_col='taxable_land_value',
    taxable_improvement_col='taxable_improvement_value',
)

# Standard report — 7 PNGs in analysis/reports/<city>/
from lvt.viz import create_city_report
create_city_report(out_df, CITY_NAME, show=False)

# Parcel map: one feature per lot (condo units summed onto their building lot)
_sum_cols = ['current_tax', 'new_tax', 'tax_change', 'taxable_land_value', 'taxable_improvement_value',
             'PHASELAND', 'PHASEBUILD', 'ASSESSMENT']
_g = df[df['geometry'].notna()].copy()
_g['_units'] = 1
_first = _g.groupby('map_ssl').agg({'PROPERTY_CATEGORY': 'first', 'OWNERNAME': 'first', 'PREMISEADD': 'first',
                                    'geometry': 'first', 'std_geoid': 'first', 'median_income': 'first',
                                    'minority_pct': 'first', 'black_pct': 'first', '_units': 'sum'})
_sums = _g.groupby('map_ssl')[_sum_cols].sum()
gdf = gpd.GeoDataFrame(_first.join(_sums).reset_index(), geometry='geometry', crs='EPSG:4326')
gdf['SSL'] = gdf['map_ssl']
_multi = gdf['_units'] > 1
gdf.loc[_multi, 'OWNERNAME'] = gdf.loc[_multi, '_units'].map(lambda n: f'{n} condo/unit accounts')
gdf.loc[_multi, 'PREMISEADD'] = gdf.loc[_multi, 'PREMISEADD'].str.replace(r'\s+#.*?(?=\s+WASHINGTON)', '', regex=True)
gdf['tax_change_pct'] = np.where(gdf['current_tax'] > 0, gdf['tax_change'] / gdf['current_tax'] * 100, np.nan)
print(f"Map features: {len(gdf):,} lots from {len(_g):,} accounts")

# Parcel-map export — GeoParquet + interactive HTML map
from lvt.parcel_map import save_parcel_map_export, create_parcel_map
map_gdf = save_parcel_map_export(
    gdf=gdf,
    city=CITY_NAME,
    output_path=f'../../analysis/maps/{CITY_NAME}.parquet',
    model_type=MODEL_TYPE,
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    parcel_id_col=PARCEL_ID_COL,
    parcel_url_template=PARCEL_URL_TEMPLATE,
    owner_name_col=OWNER_NAME_COL,
    owner_address_col=OWNER_ADDRESS_COL,
)
create_parcel_map(map_gdf, CITY_NAME)
print("Done.")

  ✓ washington_dc: 204,347 rows → ../../analysis/data/washington_dc.csv  [model: split_rate_4to1_within_class]


Map features: 126,960 lots from 204,347 accounts


  ✓ washington_dc: 126,960 parcels → ../../analysis/maps/washington_dc.parquet  [map export; 0 with record links]
  [warn] washington_dc: 126,960 parcels exceeds tile_threshold (100,000) but tippecanoe is not installed — falling back to the inline Leaflet map (this file will be large).


  ✓ washington_dc: interactive map → ../../analysis/reports/washington_dc/parcel_map.html  [126,960 parcels]
Done.


## Validation Summary and Limitations

See the executed outputs above for the figures. Key modeling choices and limitations:

- **Single levy.** DC has one real property tax, so the model covers the full property tax.
- **Revenue control.** The model base is the TY2026 billed levy; it runs about 4% above the OCFO FY2026
  cash-revenue estimate (appeal refunds, collections, dedicated revenue).
- **Within-class neutrality.** Each class (1A, 1B, 2, 3, 4, mixed use) raises exactly what it raises
  today. Burden shifts only between land-heavy and building-heavy properties *within* a class.
  Class 3/4 (vacant/blighted) parcels are mostly land already, so their bills barely change. The
  vacant-building penalty rates stay as they are.
- **Relief allocation.** Homestead, senior/disabled and cap relief are kept by giving each account the
  same proportional relief under the reform. A deduction taken entirely off the building first would
  make owner-occupant outcomes somewhat less favorable.
- **Possessory-interest (PI) accounts** (leaseholds on exempt land) have $0 land value, so they see the
  full building-rate cut.
- **Land values are OTR's.** Results depend on OTR's land/building split, which comes from neighborhood
  land models.
- **Condo units** are modeled as individual accounts (OTR assigns each a real land share, so no condo
  collapse is needed). They are merged by lot only for the map.